> **Disclaimer:** These files were generated by an LLM to help teach me about this material. There may be mistakes, but as I go through these exercises I will find them and correct them.

# Lesson 1 — The STM in the Two-Body Problem: Variational Equations, sympy, and the numpy Bookkeeping

## 1. Why the STM, restated for what you actually do

You already know the punchline conceptually: the STM $\Phi(t, t_0)$ maps a small perturbation in the initial state to the resulting perturbation in the state at time $t$:

$$\delta \mathbf{x}(t) = \Phi(t, t_0)\, \delta \mathbf{x}(t_0)$$

What I want to emphasize — because it's the part that actually changes your workflow, not just your understanding — is that $\Phi$ **is a byproduct of integration, not a separate calculation**. If you're integrating a trajectory anyway, propagating the STM alongside it costs you almost nothing extra (36 more scalar ODEs for a 6-state system) and gives you the full local sensitivity of "final state" to "initial state." That sensitivity matrix is exactly the Jacobian a Newton-Raphson shooting method needs. This is why it converges Astrogator/Copernicus so much faster when you feed them a pre-corrected guess: you've already solved the *linear* version of their BVP before handing it to them.

Most textbooks (Vallado included) introduce the STM in an orbit-determination context — mapping observation-time state uncertainty back to epoch state uncertainty for a batch filter. Same math, different purpose. We're using it for **targeting**, not estimation. Keep that distinction in your head; it'll matter when we get to weighting/regularization in Lesson 3.

## 2. The variational equations

Given $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, t)$, differentiate with respect to the initial condition $\mathbf{x}_0$:

$$\dot{\Phi}(t, t_0) = A(t)\,\Phi(t,t_0), \qquad \Phi(t_0,t_0) = I_6$$

where $A(t) = \dfrac{\partial \mathbf{f}}{\partial \mathbf{x}}\Big|_{\mathbf{x}(t)}$ is the Jacobian of the dynamics, evaluated *along the reference trajectory*. This is the part people gloss over: $A(t)$ is time-varying because it depends on $\mathbf{x}(t)$, which is why you can't just matrix-exponentiate a constant $A$ except in special cases (Clohessy-Wiltshire being the classic one you've probably used).

For 2BP in Cartesian coordinates, $\mathbf{x} = [\mathbf{r}; \mathbf{v}]$:

$$\mathbf{f}(\mathbf{x}) = \begin{bmatrix} \mathbf{v} \\ -\dfrac{\mu}{r^3}\mathbf{r} \end{bmatrix}$$

Differentiating by hand, the bottom-left block (the "gravity-gradient" block, $\partial \dot{\mathbf{v}}/\partial \mathbf{r}$) is the well-known

$$\frac{\partial \dot{\mathbf{v}}}{\partial \mathbf{r}} = -\frac{\mu}{r^3}\left(I_3 - \frac{3\,\mathbf{r}\mathbf{r}^T}{r^2}\right)$$

**Exercise 1.1** — Derive this by hand (product/chain rule on $-\mu \mathbf{r}/r^3$), then verify it against sympy's output below. Don't skip the hand derivation — it's the thing that makes you trust sympy's answer on harder problems later (CR3BP, J2, SRP) where you *can't* easily hand-check it.

In [ ]:
import sympy as sp

rx, ry, rz, vx, vy, vz, mu = sp.symbols('rx ry rz vx vy vz mu', real=True)
r_vec = sp.Matrix([rx, ry, rz])
v_vec = sp.Matrix([vx, vy, vz])
r = sp.sqrt(rx**2 + ry**2 + rz**2)

x = sp.Matrix([rx, ry, rz, vx, vy, vz])          # state vector, order matters (see §3)
f = sp.Matrix.vstack(v_vec, -mu * r_vec / r**3)   # dynamics

A = f.jacobian(x)   # this IS the variational-equations matrix, done for you

print(A[0:3, 3:6])   # top-right block -- should just be I3 (position rate = velocity)

Running this gives `Matrix([[1,0,0],[0,1,0],[0,0,1]])` for the top-right block, confirming the trivial part. The bottom-left block is the gravity-gradient tensor above — go ahead and print `A[3:6, 0:3]` and compare term-by-term to your hand derivation.

## 3. lambdify — and why you care about it specifically

`sp.jacobian()` gives you a *symbolic* 6×6 matrix full of `rx, ry, rz, mu`. You cannot hand that to `scipy.integrate.solve_ivp` — it needs a fast numeric function. `lambdify` compiles the symbolic expression into a numpy-vectorized Python function:

In [ ]:
A_func = sp.lambdify((x, mu), A, modules='numpy')
f_func = sp.lambdify((x, mu), f, modules='numpy')

Two things worth internalizing, since you'll hit both repeatedly in later lessons:

- **The signature mirrors the argument list you pass to `lambdify`, not the symbol names.** `A_func(x0, mu_val)` where `x0` is a length-6 numpy array works because sympy unpacks the `Matrix` `x` positionally — but the *return value* is a sympy-shaped object (nested list-like), not a clean numpy array. You'll almost always need `np.array(A_func(...), dtype=float)` immediately after calling it.
- **`modules='numpy'`** matters once you introduce `sp.sqrt`, `sp.sin`, trig, etc. — it tells lambdify to emit `numpy.sqrt` instead of `math.sqrt`, which is what lets the function vectorize over arrays instead of choking on them. For CR3BP (Lesson 4) this becomes non-optional.

**Verification habit worth building now:** never trust a symbolic Jacobian on a new problem without a finite-difference cross-check. It costs four lines and catches sign errors, missing chain-rule terms, or lambdify quirks before they cost you an afternoon of debugging a shooting method that mysteriously won't converge.

In [ ]:
import numpy as np

x0 = np.array([7000.0, 0.0, 0.0, 0.0, 7.5, 1.0])
mu_val = 398600.4418

def rhs_plain(t, y):
    return np.array(f_func(y, mu_val), dtype=float).flatten()

eps = 1e-6
A_fd = np.zeros((6, 6))
for j in range(6):
    dx = np.zeros(6); dx[j] = eps
    A_fd[:, j] = (rhs_plain(0, x0 + dx) - rhs_plain(0, x0 - dx)) / (2 * eps)

A_num = np.array(A_func(x0, mu_val), dtype=float).reshape(6, 6)
print(np.max(np.abs(A_num - A_fd)))   # ~1e-10, good agreement

**Exercise 1.2** — Run this. Then deliberately introduce a bug (flip a sign in the gravity term, or swap two rows of `x`) and confirm the finite-difference check catches it. Knowing what a *broken* check looks like is as valuable as knowing what a correct one looks like.

## 4. The numpy reshape mechanics (the part you asked me to slow down on)

This is the part that trips people up not because it's conceptually hard, but because it's easy to be sloppy about and get something that *runs without error* but is silently wrong.

You need to integrate 42 coupled scalar ODEs: 6 for the state, 36 for the STM. `solve_ivp` wants a single flat state vector in, a single flat derivative vector out. So you're constantly converting between:

- a $(6,6)$ matrix $\Phi$, which is how you *think* about it and how you do the matrix multiply $A\Phi$
- a flat length-36 vector, which is how `solve_ivp` stores and integrates it

The two operations you need are `.reshape(6,6)` (flat → matrix) and `.flatten()` (matrix → flat). **The critical thing:** numpy's default memory order is row-major (`order='C'`), meaning `flatten()` reads left-to-right, top-to-bottom, and `reshape` fills the same way. As long as you use the *default* order consistently on both sides, round-tripping is safe:

In [ ]:
Phi = np.eye(6)
flat = Phi.flatten()          # order='C' by default
Phi_back = flat.reshape(6, 6) # order='C' by default
assert np.allclose(Phi, Phi_back)   # fine

The bug that *will* bite you eventually: mixing `.flatten(order='F')` (Fortran/column-major — which is what you'd get if you naively ported MATLAB code, since MATLAB is column-major internally) with a default-order `.reshape()`. The array round-trips to a **transposed** matrix, not the original — no error thrown, just wrong numbers, and `Phi` for a linear dynamics matrix doesn't have any convenient symmetry that would make a transpose bug numerically obvious to you at a glance. This is exactly the kind of bug that survives code review and shows up as "my shooting method converges to something 2 km off" three weeks later.

**Rule of thumb:** pick `order='C'` (numpy's default — you don't even have to specify it) and never mix it with anything else in this codebase. `.ravel()` is `.flatten()`'s cousin that returns a view instead of a copy when possible — slightly faster, but be aware it can alias memory, which matters if you're mutating in place. For this kind of code, prefer `.flatten()`; the copy semantics are safer and the performance difference is irrelevant at this scale.

Now the augmented right-hand side:

In [ ]:
def rhs_aug(t, y, mu_val):
    x_   = y[:6]
    Phi  = y[6:].reshape(6, 6)              # flat -> matrix
    xdot = np.array(f_func(x_, mu_val), dtype=float).flatten()
    Amat = np.array(A_func(x_, mu_val), dtype=float).reshape(6, 6)
    Phidot = Amat @ Phi                     # matrix mult, NOT elementwise
    return np.concatenate([xdot, Phidot.flatten()])   # matrix -> flat

Note `Amat @ Phi` — this is genuine matrix multiplication (`@`, not `*`). This is a common typo-class bug: `*` in numpy is elementwise for arrays, and it will run without error, producing a $(6,6)$ array that is *not* $A\Phi$. Since both are $6\times6$, there's no shape mismatch to catch it. Another place a "wrong but plausible-looking" result can sneak past you silently.

Initial condition and integration:

In [ ]:
from scipy.integrate import solve_ivp

Phi0 = np.eye(6)
y0 = np.concatenate([x0, Phi0.flatten()])   # length 42

T = 2 * np.pi * np.sqrt(7000.0**3 / mu_val)   # rough period, fine for this exercise

sol = solve_ivp(rhs_aug, [0, T], y0, args=(mu_val,),
                 method='DOP853', rtol=1e-12, atol=1e-12)

Phi_T = sol.y[6:, -1].reshape(6, 6)
print(np.linalg.det(Phi_T))    # should be ~1.0

I used `DOP853` with tight tolerances deliberately — STM accuracy is more sensitive to integrator tolerance than the state trajectory itself is, because you're propagating sensitivities of sensitivities in some sense. If you get sloppy with `rtol`/`atol` here, your Newton corrector in Lesson 3 will take more iterations or fail to converge, and you'll waste time debugging the corrector when the actual problem is a loose integrator tolerance upstream.

## 5. Verification properties — your regression tests going forward

Two structural properties of $\Phi$ are worth checking every time you write a new STM propagator, because they're cheap and they catch real bugs:

**(a) $\det(\Phi) = 1$** (Liouville's theorem — phase-space volume is preserved for Hamiltonian/conservative systems). Confirmed above.

**(b) Composition property:** $\Phi(t_2, t_0) = \Phi(t_2, t_1)\,\Phi(t_1, t_0)$.

**Exercise 1.3** — Verify this numerically. I'll flag a mistake I made writing this lesson, because it's instructive: my first attempt propagated the STM continuously from $t_0$ through $t_1$ to $t_2$ in one run, then tried to split the *already-continuous* $\Phi$ history into two pieces and multiply them back together. That's wrong — the STM you pull out at the midpoint from a continuous run is $\Phi(t_1, t_0)$, but if you keep integrating *the same* $\Phi$ past $t_1$ without resetting it to identity, what you get at $t_2$ is already $\Phi(t_2,t_0)$, not $\Phi(t_2,t_1)$. To get $\Phi(t_2,t_1)$ standalone, you have to **re-initialize $\Phi$ to $I_6$ at $t_1$** and integrate a fresh leg from there. Then the product of the two independently-initialized legs matches the single continuous propagation to ~$10^{-8}$ (integrator tolerance-limited). Get this wrong and multiple shooting (Lesson 7) will silently give you the wrong Jacobian, because that method depends entirely on chaining per-segment STMs this way.

In [ ]:
# leg 1: t0 -> t1, Phi starts at I
sol1 = solve_ivp(rhs_aug, [0, T/2], y0, args=(mu_val,),
                  method='DOP853', rtol=1e-12, atol=1e-12)
Phi_10 = sol1.y[6:, -1].reshape(6, 6)          # Phi(t1, t0)

# leg 2: t1 -> t2, Phi RESET to I at t1 (this is the part to get right)
y_mid_reset = np.concatenate([sol1.y[:6, -1], np.eye(6).flatten()])
sol2 = solve_ivp(rhs_aug, [T/2, T], y_mid_reset, args=(mu_val,),
                  method='DOP853', rtol=1e-12, atol=1e-12)
Phi_21 = sol2.y[6:, -1].reshape(6, 6)          # Phi(t2, t1)

Phi_20_composed = Phi_21 @ Phi_10
print(np.max(np.abs(Phi_20_composed - Phi_T)))   # ~1e-8, matches direct propagation

## 6. Stretch exercise

**Exercise 1.4** — The 2BP Hamiltonian flow is symplectic, meaning $\Phi^T J \Phi = J$ where

$$J = \begin{bmatrix} 0_3 & I_3 \\ -I_3 & 0_3 \end{bmatrix}$$

Build $J$ with `np.block`, check this identity numerically for `Phi_T` above, and note how the residual compares to your `det = 1` check. (This is a stricter test — a matrix can have determinant 1 without being symplectic — so it's a better regression check once you're confident in it. Reserve it for a "does my new dynamics model's STM look sane" gut-check.)

In [ ]:
# Exercise 1.4 -- build J with np.block and check Phi_T.T @ J @ Phi_T == J


## Sources for this lesson
- Schaub & Junkins, *Analytical Mechanics of Space Systems*, Ch. 9 — cleanest general derivation of the variational equations I've seen; generalizes past 2BP more gracefully than most.
- Battin, *An Introduction to the Mathematics and Methods of Astrodynamics*, Ch. 9 — has the closed-form $f,g$-series STM for 2BP, which is a great *independent* analytic check on your numerically-integrated $\Phi$ once you're confident in this lesson (worth doing as an extra exercise: compare your `Phi_T` above to the closed-form result).
- Vallado, *Fundamentals of Astrodynamics and Applications*, Ch. 9 (OD chapter) — same STM, framed for estimation rather than targeting; useful to read for contrast.

## Next lesson
Lesson 2 stays in 2BP but turns this into a working single-shooting targeter — using $\Phi$'s velocity-position partition to solve "what initial $\Delta v$ gets me to this target position at this time," which is directly your maneuver-planning problem, just in the cleanest possible dynamics.